In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

In [2]:
# Create a SparkSession
spark = SparkSession.builder.appName("SparkSQL").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/02 16:26:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/02 16:26:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/02 16:26:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/02 16:26:29 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/02 16:26:29 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/02 16:26:29 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


In [5]:
def mapper(line):
    fields = line.split(',')
    return Row(ID=int(fields[0]), name=str(fields[1].encode('utf-8')), \
    age=int(fields[2]), numFriends=int(fields[3]))

In [8]:
lines = spark.sparkContext.textFile("/opt/spark/data/fakefriends.csv")
people = lines.map(mapper)

In [9]:
#Infer the schema, and register the datafrema as a table
schemaPeople = spark.createDataFrame(people).cache()
schemaPeople.createOrReplaceTempView("people")

In [10]:
#SQL can be run over DataFrames that have been registered as a table
teenagers = spark.sql("SELECT * FROM people WHERE age >= 13 and age <= 19")

In [11]:
# the results of SQL queries are RDDs and support all the normal RDD operations
for teen in teenagers.collect():
    print(teen)

Row(ID=21, name="b'Miles'", age=19, numFriends=268)
Row(ID=52, name="b'Beverly'", age=19, numFriends=269)
Row(ID=54, name="b'Brunt'", age=19, numFriends=5)
Row(ID=106, name="b'Beverly'", age=18, numFriends=499)
Row(ID=115, name="b'Dukat'", age=18, numFriends=397)
Row(ID=133, name="b'Quark'", age=19, numFriends=265)
Row(ID=136, name="b'Will'", age=19, numFriends=335)
Row(ID=225, name="b'Elim'", age=19, numFriends=106)
Row(ID=304, name="b'Will'", age=19, numFriends=404)
Row(ID=341, name="b'Data'", age=18, numFriends=326)
Row(ID=366, name="b'Keiko'", age=19, numFriends=119)
Row(ID=373, name="b'Quark'", age=19, numFriends=272)
Row(ID=377, name="b'Beverly'", age=18, numFriends=418)
Row(ID=404, name="b'Kasidy'", age=18, numFriends=24)
Row(ID=409, name="b'Nog'", age=19, numFriends=267)
Row(ID=439, name="b'Data'", age=18, numFriends=417)
Row(ID=444, name="b'Keiko'", age=18, numFriends=472)
Row(ID=492, name="b'Dukat'", age=19, numFriends=36)
Row(ID=494, name="b'Kasidy'", age=18, numFriends=194)

In [12]:
# we can also use functions instead of sql QUERIES:
schemaPeople.groupBy("age").count().orderBy("age").show()

+---+-----+
|age|count|
+---+-----+
| 18|    8|
| 19|   11|
| 20|    5|
| 21|    8|
| 22|    7|
| 23|   10|
| 24|    5|
| 25|   11|
| 26|   17|
| 27|    8|
| 28|   10|
| 29|   12|
| 30|   11|
| 31|    8|
| 32|   11|
| 33|   12|
| 34|    6|
| 35|    8|
| 36|   10|
| 37|    9|
+---+-----+
only showing top 20 rows



In [13]:
spark.stop()